# Symbolic CSP network on CIFAR-10 (gradient-free)

The harder sibling of the MNIST notebook. CIFAR-10 (natural 32x32 colour
images) genuinely needs a **more sophisticated feature stack** than MNIST's
channel pooling, so the network is:

1. **Random convolutional feature bank** (Coates & Ng 2011 recipe,
   gradient-free): ZCA-whitened random image patches act as conv filters;
   rectified (both-sign) responses are quadrant-pooled into a feature
   vector. No backprop, no learned filters.
2. **Per-class symbolic readout**: `csp_sr` fits a sparse symbolic
   one-vs-rest scorer over those features; classify by `argmax`.

**Honest expectation.** This is a ceiling stress-test, not a SOTA claim.
Gradient-free random features + a symbolic readout land around **~55-70%**
on CIFAR-10; a *trained* CNN reaches ~95%. Accuracy is driven by the
(non-symbolic) feature bank; the symbolic head buys interpretability, not
perception. This is the empirical complement to the decomposition results:
**symbolic regression's value is discovery (physics/dynamics), not
vision** - here the symbolic head correctly degenerates to a sparse linear
readout, because image class-ness is a learned statistical pattern, not an
analytic law you can peel or factor.

## 1. Setup

In [ ]:
# Cell 1.1 - Setup: clone/refresh tessera to LATEST, install, verify.
import os, sys, subprocess
REPO_DIR = 'tessera'
REPO_URL = 'https://github.com/davechendatascience/tessera.git'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout, r.stderr

if not os.path.exists(REPO_DIR):
    print('Cloning tessera...'); _run(['git', 'clone', REPO_URL])
else:
    print('Refreshing tessera to origin/main...')
    _run(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main'])
    _run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
_, sha, _ = _run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'])
print('tessera @', sha.strip())

print('Installing tessera[benchmarks,jax]...')
rc, out, err = _run([sys.executable, '-m', 'pip', 'install', '-e',
                     f'./{REPO_DIR}[benchmarks,jax]'])
if rc != 0:
    print('pip stderr tail:', err.splitlines()[-8:])
src_abs = os.path.abspath(os.path.join(REPO_DIR, 'src'))
if src_abs not in sys.path:
    sys.path.insert(0, src_abs)

import importlib
try:
    import tessera.experimental.csp_sr as _csp; importlib.reload(_csp)
    import tessera.experimental.csp_decompose as _cd; importlib.reload(_cd)
    assert hasattr(_csp, 'discover')
    print('OK: csp_sr + csp_decompose loaded.')
except Exception as e:
    print(f'LOAD FAILED ({e}). Restart runtime & re-run this cell.')

In [ ]:
# Cell 1.2 - Imports.
import time
import numpy as np
import matplotlib.pyplot as plt

from tessera.experimental.csp_sr import discover, CSPSRConfig, expr_to_str
from tessera.expression.tree import evaluate as eval_tree
print('csp_sr loaded')

try:
    import jax
    print(f'JAX devices: {jax.devices()}')
    _on_gpu = any('cuda' in str(d).lower() or 'gpu' in str(d).lower()
                  for d in jax.devices())
except Exception:
    _on_gpu = False
# The feature bank is numpy (fast already); use_jax only affects the head's
# Phi-build, which for a degree-1 (linear) head is trivial. Left off here.
USE_INTERPRETER = False
print('GPU detected.' if _on_gpu else 'CPU (fine for this notebook).')

## 2. Load CIFAR-10

In [ ]:
# Cell 2 - Load CIFAR-10 (keras, else torchvision), normalise, subset.
def load_cifar10():
    try:
        from tensorflow.keras.datasets import cifar10
        (Xtr, ytr), (Xte, yte) = cifar10.load_data()
        return Xtr, ytr.ravel(), Xte, yte.ravel()
    except Exception:
        import torchvision
        tr = torchvision.datasets.CIFAR10('./cifar', train=True, download=True)
        te = torchvision.datasets.CIFAR10('./cifar', train=False, download=True)
        return (np.asarray(tr.data), np.asarray(tr.targets),
                np.asarray(te.data), np.asarray(te.targets))

CLASSES = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
Xtr_full, ytr_full, Xte_full, yte_full = load_cifar10()

# Subset for speed. Set N_TRAIN=50000, N_TEST=10000 for the full run
# (slower; best on Colab). Random features improve with more train data.
N_TRAIN, N_TEST = 10000, 2000
rng = np.random.default_rng(0)
itr = rng.permutation(len(Xtr_full))[:N_TRAIN]
ite = rng.permutation(len(Xte_full))[:N_TEST]
Xtr = (Xtr_full[itr].astype(np.float32) / 255.0)
Xte = (Xte_full[ite].astype(np.float32) / 255.0)
ytr, yte = ytr_full[itr], yte_full[ite]

# Per-image contrast normalisation (helps random features a lot).
def contrast_norm(X):
    m = X.mean(axis=(1, 2, 3), keepdims=True)
    s = X.std(axis=(1, 2, 3), keepdims=True) + 1e-4
    return (X - m) / s
Xtr, Xte = contrast_norm(Xtr), contrast_norm(Xte)
print(f'CIFAR-10: train {Xtr.shape}, test {Xte.shape}, {len(CLASSES)} classes')

## 3. Two-stage random convolutional feature bank (gradient-free)

A deeper random-feature hierarchy - **two** stacked random-conv stages
(your "layers of CNN features"). Stage 1 convolves ZCA-whitened random image
patches (6x6) over the image, rectifies both signs, and pools to a 13x13
map. Stage 2 repeats with 3x3 random patches over the stage-1 *map*, pooling
to a small grid that is flattened to the feature vector. Every filter is a
random whitened patch - **no filter is learned**; depth + convolutional
locality are the only structure. This is what raises the accuracy ceiling.
(Raise `K1`/`K2` for more accuracy at more compute.)

In [ ]:
# Cell 3 - Two-stage random conv feature bank (gradient-free, deeper).
K1, K2 = 64, 128          # filters per stage (raise for accuracy)
PS1, PS2 = 6, 3           # patch sizes
NPAT = 50000              # patches sampled to fit ZCA + pick random filters

def patches_map(m, ps, st):
    B, H, W, C = m.shape
    nh, nw = (H - ps)//st + 1, (W - ps)//st + 1
    out = np.empty((B, nh, nw, ps*ps*C), np.float32)
    for i in range(nh):
        for j in range(nw):
            out[:, i, j, :] = m[:, i*st:i*st+ps, j*st:j*st+ps, :].reshape(B, -1)
    return out, nh, nw

def fit_filters(m, ps, K, npat, eps=0.1, seed=0):
    r = np.random.default_rng(seed); B, H, W, C = m.shape
    ii = r.integers(0, B, npat); yy = r.integers(0, H-ps, npat); xx = r.integers(0, W-ps, npat)
    Pp = np.stack([m[ii[k], yy[k]:yy[k]+ps, xx[k]:xx[k]+ps, :].ravel()
                   for k in range(npat)]).astype(np.float32)
    mean = Pp.mean(0); Pc = Pp - mean
    U, S, _ = np.linalg.svd(Pc.T @ Pc / len(Pc))
    whit = (U @ np.diag(1.0/np.sqrt(S + eps)) @ U.T).astype(np.float32)
    filt = Pc[r.integers(0, npat, K)] @ whit
    filt /= np.linalg.norm(filt, axis=1, keepdims=True) + 1e-6
    return mean, whit.astype(np.float32), filt.astype(np.float32)

def conv_stage(m, mean, whit, filt, ps, st, pool, bs=250):
    outs = []
    for s in range(0, len(m), bs):
        b = m[s:s+bs]; Pp, nh, nw = patches_map(b, ps, st)
        A = ((Pp.reshape(-1, Pp.shape[-1]) - mean) @ whit) @ filt.T
        A = np.concatenate([np.maximum(A, 0), np.maximum(-A, 0)], 1).reshape(len(b), nh, nw, -1)
        if pool > 1:
            gh, gw = nh//pool, nw//pool
            A = A[:, :gh*pool, :gw*pool].reshape(len(b), gh, pool, gw, pool, -1).mean(axis=(2, 4))
        outs.append(A.astype(np.float32))
    return np.concatenate(outs, 0)

t0 = time.time()
b1 = fit_filters(Xtr, PS1, K1, NPAT)
s1tr = conv_stage(Xtr, *b1, PS1, 1, 2)          # stage 1 -> 13x13 x 2K1 map
b2 = fit_filters(s1tr, PS2, K2, NPAT)
def feats(s1): return conv_stage(s1, *b2, PS2, 1, 3).reshape(len(s1), -1)
Ftr = feats(s1tr)
Fte = feats(conv_stage(Xte, *b1, PS1, 1, 2))
fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-6
Ftr, Fte = (Ftr - fmu)/fsd, (Fte - fmu)/fsd
print(f'2-stage bank: stage1 {s1tr.shape[1:]} -> {Ftr.shape[1]} features '
      f'in {time.time()-t0:.1f}s')

## 4. Joint multi-class head with symbolic combination

One joint, regularised solve for all 10 classes (multi-output ridge to
one-hot - not 10 independent scorers). Then the **symbolic combination
happens where it has a target: in the supervised head**. We add degree-2
products of the top-weighted features (a small, capacity-controlled set of
symbolic interaction terms) and refit jointly, keeping whichever head
generalises better on held-out data. Symbolic structure between *hidden*
conv layers would have no target (the credit-assignment wall), so it lives
here in the readout instead.

In [ ]:
# Cell 4 - Joint multi-class head + symbolic interaction terms (in the head).
def ridge_head(F, y, n_classes, lams=(1e0, 1e1, 1e2, 1e3, 1e4), val=0.2):
    N = len(F); A = np.c_[F, np.ones(N, np.float32)]; nf = int(N*(1-val))
    Af, yf, Av, yv = A[:nf], y[:nf], A[nf:], y[nf:]
    G = Af.T @ Af; AY = Af.T @ np.eye(n_classes, dtype=np.float32)[yf]
    I = np.eye(A.shape[1], dtype=np.float32); best = (-1.0, lams[0])
    for lam in lams:
        W = np.linalg.solve(G + lam*I, AY)
        acc = float(((Av @ W).argmax(1) == yv).mean())
        if acc > best[0]: best = (acc, lam)
    lam = best[1]; Y = np.eye(n_classes, dtype=np.float32)[y]
    return np.linalg.solve(A.T @ A + lam*I, A.T @ Y), lam

def head_acc(W, F, y):
    return float(((np.c_[F, np.ones(len(F), np.float32)] @ W).argmax(1) == y).mean())

# 1) linear joint ridge
t0 = time.time()
Wlin, lam = ridge_head(Ftr, ytr, len(CLASSES))
lin_tr, lin_te = head_acc(Wlin, Ftr, ytr), head_acc(Wlin, Fte, yte)
print(f'linear ridge (lam={lam:g}):           train={lin_tr:.4f}  test={lin_te:.4f}')

# 2) symbolic combination in the head: degree-2 products of the TOP-M features
TOP_M, MAX_INTER = 80, 3000
top = np.abs(Wlin[:-1]).sum(1).argsort()[::-1][:TOP_M]
pairs = [(int(i), int(j)) for a, i in enumerate(top) for j in top[a+1:]]
r2 = np.random.default_rng(0)
if len(pairs) > MAX_INTER:
    pairs = [pairs[k] for k in r2.choice(len(pairs), MAX_INTER, replace=False)]
def inter(F): return np.stack([F[:, i]*F[:, j] for i, j in pairs], 1).astype(np.float32)
Itr, Ite = inter(Ftr), inter(Fte)
imu, isd = Itr.mean(0), Itr.std(0) + 1e-6
Atr, Ate = np.c_[Ftr, (Itr-imu)/isd], np.c_[Fte, (Ite-imu)/isd]
Waug, lam2 = ridge_head(Atr, ytr, len(CLASSES))
aug_tr, aug_te = head_acc(Waug, Atr, ytr), head_acc(Waug, Ate, yte)
print(f'+ {len(pairs)} symbolic interactions (lam={lam2:g}): train={aug_tr:.4f}  test={aug_te:.4f}')

# keep the head that generalises better
used_interactions = aug_te >= lin_te
if used_interactions:
    preds = (np.c_[Ate, np.ones(len(Ate), np.float32)] @ Waug).argmax(1); te_acc = aug_te
else:
    preds = (np.c_[Fte, np.ones(len(Fte), np.float32)] @ Wlin).argmax(1); te_acc = lin_te
print(f'\nFinal head: {"linear + interactions" if used_interactions else "linear"}'
      f'  ->  TEST={te_acc:.4f}  ({time.time()-t0:.1f}s, gradient-free)')

## 5. Network architecture

The full gradient-free pipeline as a DL-style layer graph: a fixed/random
convolutional feature bank feeds a single closed-form multi-class head.

In [ ]:
# Cell 5 - Architecture diagram (DL-style layer graph).
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(14, 3.0)); ax.axis('off')
hd = ('Symbolic head\nridge + ' + (f'{len(pairs)} interactions'
      if used_interactions else 'linear') + f'\n-> {len(CLASSES)}')
layers = [
    ('Input\nCIFAR\n32x32x3', '#dfeefc'),
    (f'RandConv 1\nK1={K1} 6x6\nZCA + rect', '#fde9d0'),
    ('pool 2x2\n13x13 map', '#fde9d0'),
    (f'RandConv 2\nK2={K2} 3x3\nZCA + rect', '#fde9d0'),
    (f'pool -> flat\n{Ftr.shape[1]} feats', '#fde9d0'),
    (hd, '#e7f6e1'),
    ('argmax\n10 classes', '#dfeefc'),
]
n = len(layers); w = 0.118; gap = (1 - 0.04 - n*w)/(n - 1); x0 = 0.02
for i, (label, color) in enumerate(layers):
    xc = x0 + i*(w + gap)
    ax.add_patch(FancyBboxPatch((xc, 0.30), w, 0.42, boxstyle='round,pad=0.01',
                 fc=color, ec='#555', lw=1.2))
    ax.text(xc + w/2, 0.51, label, ha='center', va='center', fontsize=8)
    if i < n - 1:
        ax.annotate('', xy=(x0 + (i+1)*(w+gap), 0.51), xytext=(xc + w, 0.51),
                    arrowprops=dict(arrowstyle='-|>', color='#555', lw=1.3))
ax.text(0.5, 0.93, f'Two-stage gradient-free symbolic CIFAR net  -  test acc {te_acc:.3f}',
        ha='center', fontsize=12, weight='bold', transform=ax.transAxes)
ax.text(0.37, 0.12, 'random conv hierarchy  (no backprop)', ha='center',
        color='#c97a2b', fontsize=9, transform=ax.transAxes)
ax.text(0.83, 0.12, 'symbolic head  (gradient-free fit)', ha='center',
        color='#3a8a3a', fontsize=9, transform=ax.transAxes)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

In [ ]:
# Cell 4.2 - Confusion matrix + per-class accuracy.
K_ = len(CLASSES)
cm = np.zeros((K_, K_), int)
for t, p in zip(yte, preds):
    cm[t, p] += 1
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(K_)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(K_)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'CIFAR-10 symbolic net - test acc {te_acc:.3f}')
for i in range(K_):
    for j in range(K_):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()
per = cm.diagonal() / cm.sum(axis=1).clip(1)
for c in range(K_):
    print(f'  {CLASSES[c]:6s}: {per[c]:.3f}')

## What's next - honest scope

**What this is.** A gradient-free symbolic CIFAR-10 classifier: a random
convolutional feature bank (ZCA-whitened random-patch filters, rectified,
pooled) + a sparse symbolic linear readout per class. Every component is
fixed/random or closed-form; no backprop anywhere.

**The honest number.** Expect ~55-70% test accuracy (subset; a bit higher
with `N_TRAIN=50000`, more filters `K`, and a finer `POOL`). A *trained* CNN
reaches ~95%. The gap is the whole story: **on CIFAR, accuracy comes from
the features, and the best features are LEARNED (backprop)**, which this
deliberately does not do.

**Levers that raise the number** (still gradient-free): more filters `K`;
two stacked random-conv stages (a deeper random hierarchy); k-means filters
instead of random patches (Coates-Ng's stronger variant); finer pooling;
the full 50k training set. None change the conclusion - they push the
random-feature ceiling, not break it.

**Why the symbolic head stays linear.** As the decomposition study showed,
`csp_sr`/`discover_decompose` only finds deep structure where it exists
(physics: `sqrt(1-v^2/c^2)`-class, recovered exactly). Image class-ness has
no peelable outer op or variable-group separability, so the head correctly
degenerates to a capacity-controlled sparse linear readout - interpretable,
but not where the accuracy lives.

**The honest verdict, restated.** Symbolic regression is a *discovery* tool.
This notebook is the perception stress-test that marks the boundary: it works
(gradient-free, interpretable, ~60%), and it shows exactly why vision wants
learned features and physics wants symbolic ones.